# Baseline model + first submission

Goal here isn't to be clever -- it's to get *something* correct end-to-end (clean CV, correct submission format) on the leaderboard, so every later improvement has a number to beat.

Approach:
1. Drop the two known `GrLivArea` outliers from training (Id 524, 1299 -- see `01_eda.ipynb`).
2. Crude-but-safe preprocessing: median-impute numeric columns, fill categorical NaNs with the literal string `'None'` (recall most of them mean "feature absent", not "unknown"), then one-hot encode.
3. Train on `log1p(SalePrice)` since the competition scores in log space.
4. Use Ridge instead of plain `LinearRegression` -- one-hot encoding ~40 categorical columns creates ~200+ dummy columns from only ~1460 rows, which is exactly the kind of high-dimensional, collinear setup where unregularized OLS gets unstable. We'll fit both so you can see the difference in CV score directly.
5. Cross-validate in log space (matches the competition metric), then fit on all of training and predict on test.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, cross_val_score

train = pd.read_csv('../inputs/train.csv')
test = pd.read_csv('../inputs/test.csv')

# Drop the two well-known GrLivArea outliers (see 01_eda.ipynb)
train = train[~train['Id'].isin([524, 1299])].reset_index(drop=True)

print('train:', train.shape, ' test:', test.shape)

train: (1458, 81)  test: (1459, 80)


In [2]:
y_train = np.log1p(train['SalePrice'])
test_ids = test['Id']

train_features = train.drop(columns=['Id', 'SalePrice'])
test_features = test.drop(columns=['Id'])

n_train = len(train_features)
combined = pd.concat([train_features, test_features], axis=0, ignore_index=True)

numeric_cols = combined.select_dtypes(include=[np.number]).columns
categorical_cols = combined.select_dtypes(exclude=[np.number]).columns
print(f'{len(numeric_cols)} numeric columns, {len(categorical_cols)} categorical columns')

36 numeric columns, 43 categorical columns


Impute using statistics from the **training rows only** -- even though `combined` holds both train and test feature rows for consistent one-hot encoding, the fill values themselves must not be computed from test data, or we'd be leaking test-set information into preprocessing.

In [3]:
train_medians = combined.loc[:n_train - 1, numeric_cols].median()
combined[numeric_cols] = combined[numeric_cols].fillna(train_medians)
combined[categorical_cols] = combined[categorical_cols].fillna('None')

combined_encoded = pd.get_dummies(combined, columns=list(categorical_cols), drop_first=True)
print('encoded shape:', combined_encoded.shape)

X_train = combined_encoded.iloc[:n_train].reset_index(drop=True)
X_test = combined_encoded.iloc[n_train:].reset_index(drop=True)

encoded shape: (2917, 266)


In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [('LinearRegression', LinearRegression()), ('Ridge(alpha=10)', Ridge(alpha=10))]:
    scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')
    rmse = -scores
    print(f'{name:20s} log-RMSE: {rmse.mean():.4f} (+/- {rmse.std():.4f})  folds={np.round(rmse, 4)}')

LinearRegression     log-RMSE: 0.1301 (+/- 0.0139)  folds=[0.1347 0.1462 0.1239 0.1395 0.1063]


Ridge(alpha=10)      log-RMSE: 0.1154 (+/- 0.0091)  folds=[0.1245 0.108  0.1183 0.1244 0.1018]


Fit the better model (Ridge) on all of the training data and generate the submission.

In [5]:
model = Ridge(alpha=10)
model.fit(X_train, y_train)

log_preds = model.predict(X_test)
preds = np.expm1(log_preds)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
submission.to_csv('../submissions/submission_baseline.csv', index=False)
submission.head()

,Id,SalePrice
0,1461,114293.417090
1,1462,152826.707001
2,1463,179852.364336
3,1464,201273.408075
4,1465,190313.571005


## What's naive here (on purpose)

- Median-impute-everything ignores the data dictionary distinctions (e.g. `GarageYrBlt` NaN really means "no garage", not "typical year"; filling it with the median invents a fake garage age).
- `Ridge(alpha=10)` wasn't tuned -- it's a reasonable guess, not a searched value.
- No feature engineering (total square footage, house age at sale, quality x condition interactions, etc. -- all common, effective additions in this dataset).
- One-hot encoding treats ordinal quality columns (`ExterQual`: Po/Fa/TA/Gd/Ex) as unordered categories, throwing away the ordering information.

All fixable, deliberately deferred. Next step: submit `submissions/submission_baseline.csv` to Kaggle to get a real leaderboard score, then come back and tackle these one at a time -- re-running this same CV setup after each change tells us whether it actually helped before we spend a Kaggle submission on it.